# QM640 Capstone — 04_Evaluation.ipynb
## Validation, Confidence Intervals, Robustness, and Final Report Tables

This notebook converts the modeling outputs into final-report evidence. It emphasizes the evaluator's requested improvements: quantitative results, confidence intervals, target-leakage control, reliability, and transparent limitations.

In [ ]:
from pathlib import Path
import warnings, json, math, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, f, ncf, theilslopes

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             RocCurveDisplay, PrecisionRecallDisplay, mean_absolute_error,
                             mean_squared_error)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
import joblib

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
DATA_PATH = ROOT / 'data' / 'processed' / 'catastrophe_dataset.csv'
FIG_DIR = ROOT / 'figures' / 'modeling'
TABLE_DIR = ROOT / 'report_ready_tables' / 'modeling'
DOC_DIR = ROOT / 'documentation'
MODEL_DIR = ROOT / 'models'
for d in [FIG_DIR, TABLE_DIR, DOC_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT.resolve())
print('XGBoost available:', XGBOOST_AVAILABLE)

In [ ]:
df=pd.read_csv(DATA_PATH,dtype={'county_fips':str}); df['county_fips']=df['county_fips'].str.zfill(5)
df['financial_severity_log']=np.log1p(df['total_observed_loss_usd'])
thr=df['financial_severity_log'].quantile(.80)
df['high_financial_severity_flag']=(df['financial_severity_log']>=thr).astype(int)
train=df[df.year<=2018].copy(); test=df[df.year>=2019].copy()
print('Train:',len(train),'Test:',len(test),'Test high severity:',int(test.high_financial_severity_flag.sum()))

## 1. RQ1 reliability and diagnostics
The final regression uses HC3 robust standard errors. Diagnostics below quantify residual behavior, heteroscedasticity, and multicollinearity.

In [ ]:
hazard_features=['wind_speed_mph','flood_depth_feet','hail_size_inches','storm_duration_hours']
rq1=sm.OLS(df['financial_severity_log'],sm.add_constant(df[hazard_features])).fit(cov_type='HC3')
resid=rq1.resid; fitted_vals=rq1.fittedvalues
bp=het_breuschpagan(resid,sm.add_constant(df[hazard_features]))
dw=sm.stats.stattools.durbin_watson(resid)
diag=pd.DataFrame([
    ['Breusch-Pagan LM',bp[0],bp[1],'Heteroscedasticity test; HC3 used regardless'],
    ['Breusch-Pagan F',bp[2],bp[3],'Heteroscedasticity test'],
    ['Durbin-Watson',dw,np.nan,'Values near 2 indicate little first-order residual autocorrelation'],
])
diag.to_csv(TABLE_DIR/'14_RQ1_diagnostics.csv',index=False)
diag

In [ ]:
fig,ax=plt.subplots(figsize=(7,5)); ax.scatter(fitted_vals,resid,alpha=.45); ax.axhline(0,linewidth=1); ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals'); ax.set_title('RQ1 residuals vs fitted'); fig.tight_layout(); fig.savefig(FIG_DIR/'RQ1_residuals_vs_fitted.png',dpi=300,bbox_inches='tight'); plt.show()
fig=plt.figure(figsize=(7,5)); ax=fig.add_subplot(111); sm.qqplot(resid,line='45',ax=ax); ax.set_title('RQ1 residual Q–Q plot'); fig.tight_layout(); fig.savefig(FIG_DIR/'RQ1_residual_QQ.png',dpi=300,bbox_inches='tight'); plt.show()

## 2. RQ2 moderation reliability
The four pre-specified socioeconomic moderators are evaluated with a Bonferroni-adjusted alpha of 0.0125. This guards against declaring moderation based on multiple unadjusted interaction tests.

In [ ]:
work=df.copy()
for c in hazard_features+['median_income_usd','population_density_per_sq_mi','housing_age_median_years','socioeconomic_resilience_index']:
    work['z_'+c]=(work[c]-work[c].mean())/work[c].std(ddof=0)
work['hazard_intensity_index']=work[['z_'+c for c in hazard_features]].mean(axis=1)
mods=['median_income_usd','population_density_per_sq_mi','housing_age_median_years','socioeconomic_resilience_index']
rows=[]
for mod in mods:
    z='z_'+mod; inter='hazard_x_'+mod; work[inter]=work.hazard_intensity_index*work[z]
    mm=sm.OLS(work.financial_severity_log,sm.add_constant(work[['hazard_intensity_index',z,inter]])).fit(cov_type='HC3')
    ci=mm.conf_int().loc[inter]
    rows.append([mod,mm.params[inter],mm.bse[inter],ci.iloc[0],ci.iloc[1],mm.pvalues[inter],min(mm.pvalues[inter]*len(mods),1),mm.rsquared_adj])
rq2_eval=pd.DataFrame(rows,columns=['Moderator','Interaction_Coefficient','Robust_SE','CI_95_Lower','CI_95_Upper','Raw_p','Bonferroni_p','Adjusted_R_squared']).sort_values('Raw_p')
rq2_eval.to_csv(TABLE_DIR/'15_RQ2_moderation_with_adjusted_p.csv',index=False)
rq2_eval

## 3. RQ3 classification evaluation with bootstrap confidence intervals
The final evaluation uses an **independent temporal holdout (2019–2023)**. This is stricter than a random split and better reflects prospective generalization. The 95% bootstrap intervals quantify uncertainty in Accuracy, Precision, Recall, F1, ROC-AUC, and PR-AUC.

In [ ]:
num_features=['hazard_event_count','event_type_count','wind_speed_mph','hail_size_inches','flood_depth_feet','storm_duration_hours','median_income_usd','population_density_per_sq_mi','housing_age_median_years','vacancy_rate_pct','homeownership_rate_pct','pct_units_pre1980']
cat_features=['dominant_event_type','construction_type','state']; feature_cols=num_features+cat_features
X_train=train[feature_cols]; y_train=train.high_financial_severity_flag; X_test=test[feature_cols]; y_test=test.high_financial_severity_flag
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_features),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat_features)])
ratio=(len(y_train)-y_train.sum())/y_train.sum()
models={'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),'Random Forest':RandomForestClassifier(n_estimators=500,max_depth=8,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingClassifier(n_estimators=150,learning_rate=.04,max_depth=2,random_state=42)}
if XGBOOST_AVAILABLE: models['XGBoost']=XGBClassifier(n_estimators=250,max_depth=3,learning_rate=.03,subsample=.85,colsample_bytree=.85,eval_metric='logloss',random_state=42,n_jobs=2,scale_pos_weight=ratio)

def metrics(y,pred,prob):
    return {'Accuracy':accuracy_score(y,pred),'Precision':precision_score(y,pred,zero_division=0),'Recall':recall_score(y,pred,zero_division=0),'F1':f1_score(y,pred,zero_division=0),'ROC_AUC':roc_auc_score(y,prob),'PR_AUC':average_precision_score(y,prob)}

def bootstrap_ci(y,pred,prob,B=500,seed=42):
    rng=np.random.default_rng(seed); y=np.asarray(y); pred=np.asarray(pred); prob=np.asarray(prob); vals={k:[] for k in metrics(y,pred,prob)}
    n=len(y)
    for b in range(B):
        ix=rng.integers(0,n,n); yy=y[ix]
        if len(np.unique(yy))<2: continue
        mm=metrics(yy,pred[ix],prob[ix])
        for k,v in mm.items(): vals[k].append(v)
    return {k:(np.quantile(v,.025),np.quantile(v,.975)) for k,v in vals.items()}

rows=[]; preds={}
for name,model in models.items():
    pipe=Pipeline([('pre',pre),('model',model)])
    if name=='Gradient Boosting': pipe.fit(X_train,y_train,model__sample_weight=np.where(y_train==1,ratio,1.0))
    else: pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]; point=metrics(y_test,pred,prob); ci=bootstrap_ci(y_test,pred,prob)
    row={'Model':name,**point}
    for k,(lo,hi) in ci.items(): row[k+'_CI_Lower']=lo; row[k+'_CI_Upper']=hi
    rows.append(row); preds[name]=(pred,prob,pipe)
rq3_eval=pd.DataFrame(rows).sort_values('ROC_AUC',ascending=False)
rq3_eval.to_csv(TABLE_DIR/'16_RQ3_performance_with_95CI.csv',index=False)
rq3_eval

### Formal RQ3 comparison with random classification
RQ3's null hypothesis is evaluated using the 95% bootstrap confidence interval for ROC-AUC. If an ensemble model's lower confidence bound exceeds 0.50, its discrimination is considered better than random at the corresponding uncertainty level.

In [ ]:
ens=rq3_eval[rq3_eval.Model!='Logistic Regression'].copy()
ens['Lower_CI_above_0_5']=ens['ROC_AUC_CI_Lower']>0.5
ens['RQ3_Decision']=np.where(ens.Lower_CI_above_0_5,'Evidence better than random','Insufficient evidence better than random')
ens[['Model','ROC_AUC','ROC_AUC_CI_Lower','ROC_AUC_CI_Upper','F1','RQ3_Decision']].to_csv(TABLE_DIR/'17_RQ3_hypothesis_evidence.csv',index=False)
ens[['Model','ROC_AUC','ROC_AUC_CI_Lower','ROC_AUC_CI_Upper','F1','RQ3_Decision']]

### Target leakage sensitivity check
For transparency, the notebook compares the leakage-safe model against a deliberately **invalid** leaky specification that predicts the interim `high_severity_flag` using variables that were themselves used to construct `severity_metric`. The leaky result is not a valid performance estimate; it is included only to demonstrate why the final predictor exclusions are necessary.

In [ ]:
from sklearn.model_selection import train_test_split
leaky_features=['wind_speed_mph','hail_size_inches','flood_depth_feet','storm_duration_hours','log_property_damage','log_total_nfip_payout']
Xl=df[leaky_features]; yl=df.high_severity_flag
Xlt,Xlv,ylt,ylv=train_test_split(Xl,yl,test_size=.2,random_state=42,stratify=yl)
leaky=RandomForestClassifier(n_estimators=500,random_state=42,class_weight='balanced').fit(Xlt,ylt)
leaky_auc=roc_auc_score(ylv,leaky.predict_proba(Xlv)[:,1])
safe_best=rq3_eval[rq3_eval.Model!='Logistic Regression'].sort_values('ROC_AUC',ascending=False).iloc[0]
leak_compare=pd.DataFrame([
    ['Invalid/leaky interim specification','Interim high_severity_flag','Includes variables used to construct target',leaky_auc,'Not valid for final inference'],
    ['Final leakage-safe specification','High financial severity flag','Excludes all direct/derived loss variables',safe_best.ROC_AUC,'Valid temporal-holdout estimate'],
],columns=['Specification','Target','Predictor design','ROC_AUC','Interpretation'])
leak_compare.to_csv(TABLE_DIR/'18_target_leakage_sensitivity.csv',index=False)
leak_compare

In [ ]:
# Performance comparison figure with AUC confidence intervals.
plot=rq3_eval.copy(); x=np.arange(len(plot));
fig,ax=plt.subplots(figsize=(8,5)); ax.errorbar(x,plot.ROC_AUC,yerr=[plot.ROC_AUC-plot.ROC_AUC_CI_Lower,plot.ROC_AUC_CI_Upper-plot.ROC_AUC],fmt='o',capsize=5); ax.axhline(.5,linestyle='--',linewidth=1); ax.set_xticks(x); ax.set_xticklabels(plot.Model,rotation=20,ha='right'); ax.set_ylabel('ROC-AUC (95% bootstrap CI)'); ax.set_title('RQ3: Leakage-safe temporal-holdout performance'); fig.tight_layout(); fig.savefig(FIG_DIR/'RQ3_model_comparison_AUC_95CI.png',dpi=300,bbox_inches='tight'); plt.show()

## 4. RQ4 trend reliability and ARIMA holdout evaluation
With only 24 annual observations, trend inference has limited power for small effects. The final report should emphasize the Mann–Kendall confidence evidence and treat ARIMA forecasting as exploratory.

In [ ]:
annual=df.groupby('year',as_index=False).financial_severity_log.mean(); years=annual.year.to_numpy(); y=annual.financial_severity_log.to_numpy(); n=len(y)
S=sum(np.sign(y[i+1:]-y[i]).sum() for i in range(n-1)); varS=n*(n-1)*(2*n+5)/18; z=(S-1)/np.sqrt(varS) if S>0 else ((S+1)/np.sqrt(varS) if S<0 else 0); p_mk=2*(1-norm.cdf(abs(z))); tau=stats.kendalltau(years,y).statistic; sen=theilslopes(y,years,.95)

# 5-year rolling-style terminal holdout: fit 2000–2018, forecast 2019–2023. Select order on training AIC.
train_y=y[:-5]; test_y=y[-5:]; test_years=years[-5:]
rows=[]; fits={}
for p in range(4):
  for d in range(2):
    for q in range(4):
      if p==d==q==0: continue
      try:
        ft=ARIMA(train_y,order=(p,d,q),trend='t').fit(); conv=bool(ft.mle_retvals.get('converged',True)); rows.append([p,d,q,ft.aic,conv]); fits[(p,d,q)]=ft
      except: pass
cand=pd.DataFrame(rows,columns=['p','d','q','AIC','Converged']); best=cand[cand.Converged].sort_values('AIC').iloc[0]; order=tuple(int(best[c]) for c in ['p','d','q']); ft=fits[order]; pred=ft.forecast(5)
rmse=float(np.sqrt(mean_squared_error(test_y,pred))); mae=float(mean_absolute_error(test_y,pred))
rq4_eval=pd.DataFrame([{'Mann_Kendall_p':p_mk,'Kendall_tau':tau,'Sen_slope':sen.slope,'Sen_CI_Low':sen.low_slope,'Sen_CI_High':sen.high_slope,'ARIMA_holdout_order':str(order),'ARIMA_holdout_RMSE':rmse,'ARIMA_holdout_MAE':mae,'Annual_N':n}])
rq4_eval.to_csv(TABLE_DIR/'19_RQ4_trend_and_forecast_evaluation.csv',index=False)
rq4_eval

In [ ]:
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(years[:-5],train_y,marker='o',label='Training annual mean'); ax.plot(test_years,test_y,marker='o',label='Observed holdout'); ax.plot(test_years,pred,marker='o',label=f'ARIMA{order} forecast'); ax.set_xlabel('Year'); ax.set_ylabel('Mean log financial severity'); ax.set_title('RQ4: 2019–2023 ARIMA holdout evaluation'); ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR/'RQ4_ARIMA_temporal_holdout_validation.png',dpi=300,bbox_inches='tight'); plt.show()

## 5. Final quantitative summary for the report
This table is the recommended source for the final report's results chapter. It distinguishes statistically supported findings from inconclusive findings and includes the principal limitation attached to each RQ.

In [ ]:
# RQ1
rq1_dec='Reject H0' if rq1.f_pvalue<.05 else 'Fail to reject H0'
# RQ2
rq2_dec='Reject H0' if (rq2_eval.Bonferroni_p<.05).any() else 'Fail to reject H0'
# RQ3: best ensemble and CI criterion
bestens=ens.sort_values('ROC_AUC',ascending=False).iloc[0]; rq3_dec='Reject H0' if bestens.ROC_AUC_CI_Lower>0.5 else 'Fail to reject H0'
# RQ4
rq4_dec='Reject H0' if p_mk<.05 and sen.slope>0 else 'Fail to reject H0'
final_summary=pd.DataFrame([
    ['RQ1',rq1_dec,f"Adj. R²={rq1.rsquared_adj:.3f}; robust F p={rq1.f_pvalue:.2g}; wind p={rq1.pvalues['wind_speed_mph']:.3g}; flood depth p={rq1.pvalues['flood_depth_feet']:.3g}",'Outcome is an integrated county-year financial severity proxy, not event-level claim severity.'],
    ['RQ2',rq2_dec,f"Strongest interaction: {rq2_eval.iloc[0].Moderator}; coef={rq2_eval.iloc[0].Interaction_Coefficient:.3f}; Bonferroni p={rq2_eval.iloc[0].Bonferroni_p:.4g}",'Moderation is ecological at county-year level and should not be interpreted causally.'],
    ['RQ3',rq3_dec,f"Best ensemble: {bestens.Model}; AUC={bestens.ROC_AUC:.3f} (95% CI {bestens.ROC_AUC_CI_Lower:.3f}–{bestens.ROC_AUC_CI_Upper:.3f}); F1={bestens.F1:.3f}",'Strict temporal holdout and leakage exclusion reduce optimistic performance; this is intentional.'],
    ['RQ4',rq4_dec,f"Mann–Kendall p={p_mk:.3f}; tau={tau:.3f}; Sen slope={sen.slope:.4f}/year (95% CI {sen.low_slope:.4f}–{sen.high_slope:.4f}); ARIMA holdout RMSE={rmse:.3f}",'Only 24 annual points; adequate for large trends but limited power for small trends.'],
],columns=['RQ','Hypothesis decision','Quantitative evidence','Reliability / interpretation caveat'])
final_summary.to_csv(TABLE_DIR/'20_FINAL_results_summary_by_RQ.csv',index=False)
final_summary

## 6. Final-report wording guardrails
- Use **"integrated county-year financial severity"** or **"financial-severity proxy"** when referring to the combined NOAA + NFIP outcome.
- Reserve **"NFIP claim severity"** for NFIP-specific payout-per-claim sensitivity analysis.
- State that NOAA and NFIP are matched by county and year, not by event/claim transaction; therefore non-flood NOAA hazards cannot be asserted to have caused NFIP payouts.
- Do not present the deliberately leaky sensitivity model as predictive evidence.
- Report both point estimates and uncertainty (95% CIs/p-values), and distinguish statistical significance from business importance.